In [1]:
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import cv2, os, shutil, glob, json
import numpy as np
import pandas as pd
import torch.nn as nn
import seaborn as sns
from pathlib import Path
pd.set_option('display.max_columns', None)

In [2]:
OPTIONS = json.loads(open('../../../Task/info.json', 'r').read())
OPTIONS

{'img_size': [32, 128, 128],
 'step': 3,
 'model': 'segresnet',
 'lr': 0.0001,
 'loss': 'dice_focal',
 'batch': 2,
 'scheduler': 'plateau'}

In [3]:
IMG_SIZE = tuple(OPTIONS.get('img_size'))
IMG_SIZE

(32, 128, 128)

In [4]:
def loadFile(path, size=(128, 128, 128)):
    file = Path(path)
    ext  = file.suffix.lower()

    if ext == '.png':
        return cv2.imread(path)
    
    if ext == '.npy':
        return np.load(path)
    
    if ext == '.dat':
        return np.reshape(np.fromfile(path, dtype=np.single), size)

    return None

def getFiles(path, limit=None, shuffle=False):
    target = sorted(glob.glob(os.path.join(path, '*')))
    if shuffle:
        np.random.shuffle(target) 
    return target[:limit]

In [5]:
images = [loadFile(img, IMG_SIZE) for img in getFiles('images')]
masks  = [loadFile(img, IMG_SIZE) for img in getFiles('masks')]

IMG_SIZE = images[0].shape[0]
print(len(images), IMG_SIZE)
images[:5]

2805 32


[array([[[0.5026313 , 0.5026313 , 0.5026313 , ..., 0.48177254,
          0.48243746, 0.4811131 ],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.39019722,
          0.38500077, 0.3870629 ],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.34626895,
          0.3458204 , 0.3454912 ],
         ...,
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.472669  ,
          0.4755251 , 0.47828618],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.49668372,
          0.49909437, 0.50123376],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.5066623 ,
          0.5098656 , 0.5109614 ]],
 
        [[0.5026313 , 0.5026313 , 0.5026313 , ..., 0.48429793,
          0.4834892 , 0.48048073],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.38806704,
          0.3845797 , 0.38495204],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.34511763,
          0.3444294 , 0.34667778],
         ...,
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.47650492,
          0.47683808, 0.

In [6]:
def setFolder(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)

setFolder('../../../Model/Database/Target/images')
setFolder('../../../Model/Database/Target/masks')

for i, (img, mask) in enumerate(zip(images, masks)):
    np.save(f'../../../Model/Database/Target/images/img_{i:04d}.npy', img)
    np.save(f'../../../Model/Database/Target/masks/img_{i:04d}.npy', mask)